# Build an MPLS L3VPN Knowledge Graph — in Python

A tiny, **runnable** companion to the OSPF knowledge-graph course. Same moves, one
layer deeper: instead of areas and prefixes, we model an **MPLS L3VPN** — a customer
service riding a **transport LSP** stitched across the provider core.
It uses **Python + networkx**, so it runs right here in Colab — no database to install.

**The one idea:** a VPN service does not travel over "the network" — it travels inside a
specific **label-switched path (LSP)** built across core (P) routers over the IGP underlay.
Break any hop on that path and the VPN loses its path, even though the P router that failed
has *no customer config on it at all*. The graph needs one node MPLS makes essential: the
**LSP** the customer is riding.

**Is this machine learning?** No. Just structured facts you can question — no training, no
guessing. If you can read an MPLS forwarding table, you can read this.


## The words you need

- **PE** — provider *edge* router: where the customer plugs in, where the VRF lives.
- **P** — provider *core* router: only swaps labels; holds no customer VRFs.
- **LSP** — label-switched path: the transport *tunnel* a VPN rides. Up or down.
- **VRF** — a customer's private routing table on a PE (their slice of the L3VPN).
- **Service** — the customer business service that runs inside the VRF.
- **edge property** — a fact on a link, e.g. a transit hop's `state='down'`.
- **traversal** — walking the links to answer a question (that is the blast radius).


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# networkx and matplotlib come pre-installed in Colab.
# A DiGraph is a *directed* graph: every edge has a direction, like our arrows.
G = nx.DiGraph()
print('empty graph ready:', G)

## Step 1 — Create your nodes

Two edge routers (**PE**), two core routers (**P**), one transport **LSP**, one customer
**VRF**, and the **Service** that rides it. Each node gets a `label` (its type).


In [ ]:
# edge (PE) and core (P) routers
G.add_node('pe-west',  label='PE')
G.add_node('pe-east',  label='PE')
G.add_node('p-core-1', label='P')
G.add_node('p-core-2', label='P')

# the transport LSP (a tunnel), the customer VRF, and the business service
G.add_node('lsp-blue',      label='LSP', state='up')
G.add_node('vrf-acme',      label='VRF')
G.add_node('Acme Payments', label='Service', criticality='critical')

print(G.number_of_nodes(), 'nodes:', list(G.nodes))

## Step 2 — Wire the overlay: service -> VRF -> PE, and VRF -> LSP

The **Service** `RUNS_IN` the **VRF**, which sits `ON` a **PE** and `USES_LSP` the transport
**LSP**. This is the overlay: the customer's world, terminated at the edge.


In [ ]:
G.add_edge('Acme Payments', 'vrf-acme', rel='RUNS_IN')
G.add_edge('vrf-acme', 'pe-east',  rel='ON')
G.add_edge('vrf-acme', 'lsp-blue', rel='USES_LSP')
G.add_edge('pe-east',  'lsp-blue', rel='HAS_LSP')

for u, v, d in G.edges(data=True):
    print(f'{u} --{d["rel"]}--> {v}')

## Step 3 — Wire the underlay: the LSP transits the core, hop by hop

The LSP is label-switched across each **P** router. We store each hop's health right on the
edge as a `state` property. The middle hop has **lost its labels** (`state='down'`) — and one
dead hop is enough to break the whole path.


In [ ]:
G.add_edge('lsp-blue', 'p-core-1', rel='TRANSITS', state='up')
G.add_edge('lsp-blue', 'p-core-2', rel='TRANSITS', state='down')   # <-- the failure

for u, v, d in G.edges(data=True):
    if d.get('rel') == 'TRANSITS':
        print(f'{u} transits {v}: {d["state"]}')
print('graph now has', G.number_of_nodes(), 'nodes and', G.number_of_edges(), 'edges')

## See it

Colour the nodes by their label; draw the down transit red and dashed.


In [ ]:
colors = {'PE': '#0f7f74', 'P': '#3aa0ff', 'LSP': '#7d5bd0', 'VRF': '#9aa5ad', 'Service': '#c8702a'}
node_colors = [colors.get(G.nodes[n].get('label'), '#cccccc') for n in G]
pos = nx.spring_layout(G, seed=7)   # seed keeps the layout stable

plt.figure(figsize=(9, 6))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1900, alpha=0.92)
nx.draw_networkx_labels(G, pos, font_size=8)

down  = [(u, v) for u, v, d in G.edges(data=True) if d.get('state') == 'down']
other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in down]
nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=16, edge_color='#8894a0')
nx.draw_networkx_edges(G, pos, edgelist=down,  arrows=True, arrowsize=16, edge_color='#d34b3f', style='dashed')
nx.draw_networkx_edge_labels(G, pos, font_size=7,
    edge_labels={(u, v): d['rel'] for u, v, d in G.edges(data=True)})

plt.axis('off'); plt.title('MPLS L3VPN knowledge graph'); plt.tight_layout(); plt.show()

## Step 4 — Ask the blast-radius question

*If a transport LSP loses a hop, which VPN customers lose their path?*

We **traverse**: a broken LSP (its own `state='down'`, or any `TRANSITS` hop down) ->
the VRFs that ride it -> the services inside those VRFs. Notice the failing P router
has no customer config on it at all — yet the graph names the customer anyway.


In [ ]:
def blast_radius(G):
    """Services put at risk by any broken transport LSP.

    An LSP is broken if its own state is 'down', OR if any hop it TRANSITS is 'down'.
    Returns (lsp, failed_hop, pe, vrf, service) tuples.
    """
    at_risk = []
    for lsp in G:
        if G.nodes[lsp].get('label') != 'LSP':
            continue
        # 1) is this LSP broken? collect any down hops
        down_hops = [v for _, v, d in G.out_edges(lsp, data=True)
                     if d.get('rel') == 'TRANSITS' and d.get('state') == 'down']
        lsp_down = G.nodes[lsp].get('state') == 'down' or bool(down_hops)
        if not lsp_down:
            continue
        failed_hop = down_hops[0] if down_hops else '(lsp state down)'
        # 2) VRFs riding this LSP
        for vrf, _, d2 in G.in_edges(lsp, data=True):
            if d2.get('rel') != 'USES_LSP':
                continue
            # which PE the VRF sits on
            pe = next((p for _, p, d in G.out_edges(vrf, data=True) if d.get('rel') == 'ON'), '?')
            # 3) services running in that VRF
            for svc, _, d3 in G.in_edges(vrf, data=True):
                if d3.get('rel') == 'RUNS_IN':
                    at_risk.append((lsp, failed_hop, pe, vrf, svc))
    return at_risk

for lsp, hop, pe, vrf, svc in blast_radius(G):
    print(f'{lsp} broken at {hop}  ->  {pe} / {vrf}  ->  AT RISK: {svc}')

The forwarding table only ever showed you a missing label on `p-core-2`. Your graph just
told you **Acme Payments** is at risk — because you gave it the LSP node and wired the VPN
that rides it.


## What-if: repair the hop, then break it again

Flip the down transit to `up` and ask again — nothing is at risk, the LSP is whole. Set it
back to `down` — Acme Payments returns. You are running "what if this core hop fails?" on a
model, safely.


In [ ]:
G['lsp-blue']['p-core-2']['state'] = 'up'      # repair the hop
print('after repair: ', blast_radius(G) or 'nothing at risk')

G['lsp-blue']['p-core-2']['state'] = 'down'    # break it again
print('after re-break:', [svc for *_, svc in blast_radius(G)])

## Make it yours

Change the five values below to **one** VRF and **one** service *you* run, riding **one**
transport LSP, then re-run. Your own service name comes back from `blast_radius(G)` — that
is the moment the tool is yours. (Model the smallest slice that answers one real question;
add a backup LSP or a second VRF later, one node at a time.)


In [ ]:
# --- change these five values to your VPN ---
MY_PE      = 'pe-lon-1'
MY_P       = 'p-ams-2'
MY_LSP     = 'lsp-green'
MY_VRF     = 'vrf-globex'
MY_SERVICE = 'Globex Ledger'
# --------------------------------------------

G.add_node(MY_PE,      label='PE')
G.add_node(MY_P,       label='P')
G.add_node(MY_LSP,     label='LSP', state='up')
G.add_node(MY_VRF,     label='VRF')
G.add_node(MY_SERVICE, label='Service', criticality='critical')

G.add_edge(MY_SERVICE, MY_VRF, rel='RUNS_IN')
G.add_edge(MY_VRF, MY_PE,  rel='ON')
G.add_edge(MY_VRF, MY_LSP, rel='USES_LSP')
G.add_edge(MY_PE,  MY_LSP, rel='HAS_LSP')
G.add_edge(MY_LSP, MY_P,   rel='TRANSITS', state='down')   # the modelled failure

for lsp, hop, pe, vrf, svc in blast_radius(G):
    print(f'{lsp} broken  ->  AT RISK: {svc}')

## You built an MPLS knowledge graph

From an empty graph to a question no forwarding table can answer. The shapes you used —
*a service runs in a VRF, a VRF rides an LSP, an LSP transits the core* — are the same graph
moves as the OSPF lab, one layer down.

**Where next:** that transport LSP is built *over the IGP underlay* — the OSPF graph you built
first. Chain them and one graph answers a question that spans both layers: an OSPF area going
dark breaks the LSP that a VPN rides that a customer's payments depend on. The companion Neo4j
lab does this exact model in **Cypher** against a real graph database — same nodes, same edges,
same blast-radius question.
